# KG1 v150 MASTER - Plano DEFINITIVO (research devastador 6 agentes + DeepSeek-R1)

## Estado real confirmado 2026-04-25:
- Score Felipe: **0.86** (rank #330)
- TOP 1: 0.87 (MOONMOON)
- Slots Kaggle hoje: **5/5 disponiveis**
- A100 80GB ALTA RAM (Colab Pro+)

## DESCOBERTAS DEVASTADORAS:

🔥 **huikang (Tong Hui Kang) PUBLICOU TUDO publicamente**:
- Pipeline GitHub: `tonghuikang/nemotron`
- Adapter Kaggle: `andreyyunoshev/huikang-nemotron-adapter-mirror` (1.4GB)
- Solution + traces: `samvalladares/huikang-nemotron-artifacts` (1.5GB)
- Corpus: https://nemotron.huikang.dev/corpus.html
- Hyperparams: rank=32, train_mlp+attn+**unembed (lm_head!)**, batch=64, max_seq=8192, **lr=2e-5 StepLinearDecay** (NAO 2e-4!), 1 epoch

🔥 **GRPO 20x speedup** (Komil Parmar): transformers>=5.3.0 + REMOVE trust_remote_code + gradient_checkpointing=False -> 2 tok/s -> 38 tok/s

🔥 **uditjain13 expert adapters** prontos para DARE-TIES merge:
- `uditjain13/nemotron-30b-math-peft`
- `uditjain13/nemotron-30b-code-peft`
- `uditjain13/nemotron-30b-science-peft`

🔥 **Distillation Gemini OK** (CPMP staff): "distilling Gemini Flash 2.0 outputs looks fine"

## 3 MODOS (escolha no form param):

### MODE = "huikang_only" (30 min, baseline)
Submit huikang adapter direto. Expected: **0.85**.

### MODE = "dare_ties_merge" (1.5h, RECOMENDADO)
DARE-TIES merge huikang + uditjain13 expert adapters (math, code, science).
Weights: 0.4 huikang + 0.2 each domain. Density=0.3, scaling=0.7.
Expected: **0.85-0.88** (25-35% prob 0.87+).

### MODE = "train_warmstart" (3-4h, max gain)
Warmstart huikang adapter + continue Tong recipe training (lr=2e-5 StepDecay, 0.5 epoch, batch=64 effective).
Adiciona cryptarithm 813 + eq_guess 150 + synth 4425.
Expected: **0.85-0.89** (15-40% prob 0.87+).

## Setup

1. Runtime -> A100 ALTA RAM (80 GB)
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. Aceitar termos: https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
4. Run all


In [ ]:
# CELL UNICA v150 MASTER - 3 MODOS (huikang_only / dare_ties_merge / train_warmstart)
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil

# === FORM PARAMS ===
MODE = "dare_ties_merge"  #@param ["huikang_only", "dare_ties_merge", "train_warmstart"]
DRY_RUN = False  #@param {type:"boolean"}

# === ENV configs ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print('=' * 70)
print(f'KG1 v150 MASTER - MODE: {MODE}')
print('=' * 70)

# ============================================================
# STEP 1: Setup secrets + GPU + Kaggle CLI
# ============================================================
print('\n[1/9] Setup secrets + GPU + Kaggle CLI')
try:
    from google.colab import userdata
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                print(f'  HF token via {k} ({len(v)} chars)')
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing - configure Colab Secret')

    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print(f'  Kaggle: {os.environ["KAGGLE_USERNAME"]}')
    except Exception as e:
        print(f'  [WARN] Kaggle creds: {e}')
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# ============================================================
# STEP 2: Install dependencies (transformers >= 5.3.0!)
# ============================================================
print('\n[2/9] Install dependencies (transformers>=5.3.0 - Komil fix)')
deps = [
    'transformers>=5.3.0',  # FIX Komil: GRPO 20x speedup
    'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'kaggle', 'einops', 'packaging', 'ninja', 'triton',
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

for m in ['transformers', 'peft', 'trl', 'bitsandbytes']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Install mamba-ssm via GIT (PyPI quebrado torch 2.10+)
# ============================================================
print('\n[3/9] Install mamba-ssm + causal-conv1d (GIT source)')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'
print(f'  Detected: {py_ver} | torch{torch_short} | abi{abi_str}')

# Try wheels first (fast), fallback to git
attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

success = False
for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] wheel mamba_ssm 2.3.1 + causal_conv1d 1.6.1')
        success = True
        break

if not success:
    print('  [WARN] wheels falharam, fallback git source...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                   'git+https://github.com/Dao-AILab/causal-conv1d.git@main'], timeout=900)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                   'git+https://github.com/state-spaces/mamba.git@main'], timeout=1500)

for m in ['mamba_ssm', 'causal_conv1d']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 4: Download huikang adapter (Kaggle dataset)
# ============================================================
print('\n[4/9] Download huikang adapter from Kaggle dataset')
HUIKANG_DIR = '/content/huikang_adapter'
os.makedirs(HUIKANG_DIR, exist_ok=True)

# Try andreyyunoshev mirror first (1.4GB, lightweight - just adapter)
kaggle_dataset = 'andreyyunoshev/huikang-nemotron-adapter-mirror'
print(f'  Downloading {kaggle_dataset}...')
r = subprocess.run(['kaggle', 'datasets', 'download', '-d', kaggle_dataset,
                   '-p', HUIKANG_DIR, '--unzip'], capture_output=True, text=True, timeout=600)
if r.returncode != 0:
    print(f'  [WARN] mirror failed: {r.stderr[:200]}')
    print('  Trying alternative samvalladares/huikang-nemotron-artifacts...')
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d', 'samvalladares/huikang-nemotron-artifacts',
                       '-p', HUIKANG_DIR, '--unzip'], capture_output=True, text=True, timeout=900)
    if r.returncode != 0:
        raise RuntimeError(f'Both Kaggle datasets failed: {r.stderr[:300]}')

# Find adapter_config.json
adapter_path = None
for root, dirs, files in os.walk(HUIKANG_DIR):
    if 'adapter_config.json' in files:
        adapter_path = root
        break
if not adapter_path:
    raise RuntimeError(f'adapter_config.json not found in {HUIKANG_DIR}')
print(f'  [OK] Adapter at: {adapter_path}')

with open(os.path.join(adapter_path, 'adapter_config.json')) as f:
    adapter_cfg = json.load(f)
print(f'  Config: r={adapter_cfg.get("r")}, alpha={adapter_cfg.get("lora_alpha")}, targets={str(adapter_cfg.get("target_modules"))[:80]}...')

# ============================================================
# STEP 5: MODE huikang_only -> validate + submit
# ============================================================
if MODE == 'huikang_only':
    print('\n[5/9] MODE huikang_only - prepare submission ZIP')
    SUBMIT_DIR = '/content/submit_v150_huikang'
    os.makedirs(SUBMIT_DIR, exist_ok=True)

    # Copy adapter_config + safetensors only (Kaggle gate strict)
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        src = os.path.join(adapter_path, fname)
        if os.path.exists(src):
            shutil.copy(src, SUBMIT_DIR)
            print(f'  Copied {fname}: {os.path.getsize(src)/1e6:.1f} MB')

    # Zip
    zip_path = '/content/v150_huikang_submit.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)
    sz_mb = os.path.getsize(zip_path) / 1e6
    print(f'  ZIP: {zip_path} ({sz_mb:.1f} MB)')

    if not DRY_RUN:
        msg = 'v150 MASTER MODE huikang_only - direct adapter from Progress Prize winner'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout}{r.stderr}'[:500])
    else:
        print(f'  [DRY_RUN] Would submit: {zip_path}')

    print('\n' + '=' * 70)
    print(f'MODE huikang_only DONE. Expected: 0.85')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# STEP 5b: For dare_ties or train, load model
# ============================================================
print(f'\n[5/9] Load Nemotron-30B BF16 (MODE={MODE})')
from huggingface_hub import HfApi, snapshot_download
whoami = HfApi(token=hf_token).whoami()
print(f'  HF user: {whoami["name"]}')

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'  Loading model 30B BF16...')
t0 = time.time()
# NOTE: trust_remote_code=True for safety, gradient_checkpointing handled later
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    token=hf_token,
    attn_implementation='eager',  # safest for Nemotron-H
)
model.config.use_cache = False
print(f'  [OK] Model loaded in {time.time()-t0:.1f}s')
print(f'  VRAM after model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ============================================================
# STEP 6: MODE dare_ties_merge
# ============================================================
if MODE == 'dare_ties_merge':
    print('\n[6/9] MODE dare_ties_merge - load adapters + merge DARE-TIES')

    # Load huikang as primary
    print(f'  Loading huikang adapter from {adapter_path}...')
    model = PeftModel.from_pretrained(model, adapter_path, adapter_name='huikang', token=hf_token)

    # Load uditjain13 expert adapters
    expert_adapters = {
        'math': 'uditjain13/nemotron-30b-math-peft',
        'code': 'uditjain13/nemotron-30b-code-peft',
        'science': 'uditjain13/nemotron-30b-science-peft',
    }

    loaded_adapters = ['huikang']
    for name, repo in expert_adapters.items():
        try:
            print(f'  Loading {name} from {repo}...')
            local = snapshot_download(repo_id=repo, token=hf_token)
            model.load_adapter(local, adapter_name=name)
            loaded_adapters.append(name)
        except Exception as e:
            print(f'  [WARN] {name}: {e}')

    print(f'  Loaded adapters: {loaded_adapters}')

    # DARE-TIES merge
    weights_map = {'huikang': 0.4, 'math': 0.2, 'code': 0.2, 'science': 0.2}
    final_adapters = [a for a in loaded_adapters]
    final_weights = [weights_map.get(a, 0.1) for a in final_adapters]
    # Renormalize
    s = sum(final_weights)
    final_weights = [w/s for w in final_weights]

    print(f'  Merging via dare_ties: {dict(zip(final_adapters, final_weights))}')
    model.add_weighted_adapter(
        adapters=final_adapters, weights=final_weights, adapter_name='merged',
        combination_type='dare_ties', density=0.3,
    )
    model.set_adapter('merged')

    # Save merged
    OUTPUT_DIR = '/content/output_v150_merged'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
    model.save_pretrained(ADAPTER_DIR, selected_adapters=['merged'])
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f'  [OK] Merged adapter saved at {ADAPTER_DIR}')

    # Submit
    print('\n[8/9] Prepare Kaggle submission ZIP')
    SUBMIT_DIR = '/content/submit_v150_merged'
    os.makedirs(SUBMIT_DIR, exist_ok=True)
    for fname in os.listdir(ADAPTER_DIR):
        if fname in ('adapter_config.json', 'adapter_model.safetensors'):
            shutil.copy(os.path.join(ADAPTER_DIR, fname), SUBMIT_DIR)
            print(f'  Copied {fname}: {os.path.getsize(os.path.join(SUBMIT_DIR, fname))/1e6:.1f} MB')

    zip_path = '/content/v150_merged_submit.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)
    print(f'  ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')

    if not DRY_RUN:
        msg = 'v150 MASTER MODE dare_ties_merge - huikang+math+code+science DARE-TIES'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout}{r.stderr}'[:500])

    # Upload HF
    try:
        from huggingface_hub import create_repo
        DEST_REPO = 'felipesp1983/kg1-v150-dare-ties-merged'
        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
        HfApi(token=hf_token).upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST_REPO,
                                            commit_message='v150 DARE-TIES merge huikang+experts')
        print(f'  [OK] HF: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'  [WARN] HF upload: {e}')

    print('\n' + '=' * 70)
    print(f'MODE dare_ties_merge DONE. Expected: 0.85-0.88')
    print('=' * 70)
    raise SystemExit(0)

# ============================================================
# STEP 6: MODE train_warmstart
# ============================================================
if MODE == 'train_warmstart':
    print('\n[6/9] MODE train_warmstart - warmstart huikang + Tong recipe continue')

    # Enable input grads + gradient_checkpointing
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})

    # Load huikang as warmstart, then merge to base, then add new LoRA
    print(f'  Loading huikang adapter as warmstart...')
    model = PeftModel.from_pretrained(model, adapter_path, adapter_name='huikang',
                                      token=hf_token, is_trainable=True)
    print(f'  [OK] Warmstart loaded')
    print(f'  VRAM after warmstart LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.3f}%)')

    # Download datasets
    print('\n  Downloading additional datasets...')
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    URLS = {
        'tong.csv': 'https://huggingface.co/datasets/felipesp1983/nemotron-cot-tong-dgxchen/resolve/main/less_cot.csv',
        'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
        'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
        'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
    }
    for name, url in URLS.items():
        out = os.path.join(DATA_DIR, name)
        urllib.request.urlretrieve(url, out)
        print(f'  [OK] {name}: {os.path.getsize(out)/1e6:.2f} MB')

    # Format datasets (Tong format)
    from datasets import Dataset
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
    all_data = []

    with open(os.path.join(DATA_DIR, 'tong.csv'), encoding='utf-8') as f:
        rd = csv.DictReader(f)
        for row in rd:
            p, c = (row.get('prompt') or '').strip(), (row.get('generated_cot') or '').strip()
            if p and c:
                all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': 'tong'})

    for fname, src_name in [('cryptarithm_813.jsonl', 'crypt_813'),
                              ('eq_guess_150.jsonl', 'eq_150'),
                              ('synth_4425.jsonl', 'synth')]:
        with open(os.path.join(DATA_DIR, fname), encoding='utf-8') as f:
            for line in f:
                row = json.loads(line)
                p = (row.get('prompt') or '').strip()
                c = (row.get('cot') or row.get('generated_cot') or '').strip()
                if p and c and (row.get('is_valid', True) if 'is_valid' in row else True):
                    all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': src_name})

    ORDER = {'tong': 0, 'eq_150': 1, 'crypt_813': 2, 'synth': 3}
    all_data.sort(key=lambda x: ORDER.get(x['src'], 99))
    ds = Dataset.from_list(all_data)
    print(f'  Total samples: {len(ds)}')

    # Tokenize max_seq=4096
    print('\n[7/9] Tokenize (max_seq=4096)')
    MAX_LENGTH = 4096

    def fmt(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['completion']}]
        full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_ids = tokenizer(full, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)['input_ids']
        prm_ids = tokenizer(prm, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)['input_ids']
        labels = list(full_ids)
        for i in range(min(len(prm_ids), len(labels))):
            labels[i] = -100
        return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

    tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
    seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
    print(f'  Lengths: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

    # Train Tong recipe (lr=2e-5 StepDecay, 0.5 epoch warmstart)
    print('\n[8/9] TRAIN Tong recipe warmstart (lr=2e-5 StepDecay, 0.5 epoch)')
    gc.collect(); torch.cuda.empty_cache()

    OUTPUT_DIR = '/content/output_v150_warmstart'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id
    def collator(features):
        max_len = max(len(f['input_ids']) for f in features)
        max_len = ((max_len + 7) // 8) * 8
        ids, masks, lbls = [], [], []
        for f in features:
            pad = max_len - len(f['input_ids'])
            ids.append(f['input_ids'] + [PAD_ID]*pad)
            masks.append(f['attention_mask'] + [0]*pad)
            lbls.append(f['labels'] + [-100]*pad)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(masks), 'labels': torch.tensor(lbls)}

    from transformers import Trainer, TrainingArguments
    N_STEPS = max(50, math.ceil(len(tokenized) / 64) // 2)  # 0.5 epoch warmstart
    WARMUP = max(5, int(N_STEPS * 0.05))

    train_args = TrainingArguments(
        output_dir=OUTPUT_DIR, num_train_epochs=1,
        max_steps=N_STEPS,                        # 0.5 epoch warmstart
        per_device_train_batch_size=1,
        gradient_accumulation_steps=64,           # Tong batch_effective=64
        learning_rate=2e-5,                       # TONG StepLinearDecay default
        lr_scheduler_type='linear',               # StepLinear approximation
        warmup_steps=WARMUP,
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.0,                         # Tong: 0.0
        max_grad_norm=1.0,
        logging_steps=10,
        save_steps=max(20, N_STEPS // 4),
        save_total_limit=3,
        bf16=True,
        optim='adamw_torch_fused',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
        remove_unused_columns=False, report_to='none',
        dataloader_num_workers=0,
        seed=42,
    )

    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator)
    print(f'  Steps: {N_STEPS} | Warmup: {WARMUP}')
    t0 = time.time()
    trainer.train()
    print(f'  Training: {(time.time()-t0)/60:.1f} min')

    # Save
    ADAPTER_DIR = f'{OUTPUT_DIR}/final_adapter'
    trainer.save_model(ADAPTER_DIR)
    print(f'  [OK] Saved at {ADAPTER_DIR}')

    # Submit
    SUBMIT_DIR = '/content/submit_v150_warmstart'
    os.makedirs(SUBMIT_DIR, exist_ok=True)
    for fname in ('adapter_config.json', 'adapter_model.safetensors'):
        src = os.path.join(ADAPTER_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, SUBMIT_DIR)

    zip_path = '/content/v150_warmstart_submit.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

    if not DRY_RUN:
        msg = 'v150 MASTER MODE train_warmstart - huikang warmstart + Tong recipe 0.5ep'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout}{r.stderr}'[:500])

    # Upload HF
    try:
        from huggingface_hub import create_repo
        DEST_REPO = 'felipesp1983/kg1-v150-warmstart'
        create_repo(DEST_REPO, private=False, exist_ok=True, token=hf_token)
        HfApi(token=hf_token).upload_folder(folder_path=ADAPTER_DIR, repo_id=DEST_REPO,
                                            commit_message='v150 warmstart Tong recipe')
        print(f'  [OK] HF: https://huggingface.co/{DEST_REPO}')
    except Exception as e:
        print(f'  [WARN] HF upload: {e}')

    print('\n' + '=' * 70)
    print(f'MODE train_warmstart DONE. Expected: 0.85-0.89')
    print('=' * 70)
